# Orpheus TTS LoRA fine-tune — cool-jahns (Jeremy Jahns)

**Run on Colab with a T4 GPU.** Fine-tunes `unsloth/orpheus-3b-0.1-ft` (Llama-3.2-3B backbone) on the **408 pre-sliced segments**, then exports a **Q4 GGUF** that runs on your M1 (verified 2.36 GB, LM Studio Metal).

> **Honest notes:**
> 1. Authored on a Mac (no GPU), not test-run — run a cell, paste errors back, the assistant fixes them.
> 2. The **SNAC audio-token formatting** (cell 6) is the intricate part and its token IDs/format track Unsloth's recipe, which changes. If cell 6/7 misbehave, open Unsloth's **official Orpheus-TTS notebook** ([docs.unsloth.ai → Text-to-Speech](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning)) and swap its dataset for `./jahns_ds` built in cell 5 — everything else is standard.

In [ ]:
# 1. GPU check
!nvidia-smi

In [ ]:
# 2. Install
!pip install -q unsloth snac faster-whisper soundfile librosa datasets
# (unsloth pulls a matched torch/transformers/trl/peft set)

In [ ]:
# 3. Get the 408 segments. Upload jahns-segments-24k.zip to Drive first.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/seg
!unzip -o -q /content/drive/MyDrive/jahns-segments-24k.zip -d /content/seg
import glob
wavs = sorted(glob.glob('/content/seg/segments/*.wav'))
print(len(wavs), 'segments')

In [ ]:
# 4. ASR each segment (Faster-Whisper) -> transcript. ~5-10 min on T4.
from faster_whisper import WhisperModel
import os, json
asr = WhisperModel('large-v3', device='cuda', compute_type='float16')
rows = []
for i, w in enumerate(wavs):
    segs, _ = asr.transcribe(w, language='en', beam_size=5)
    text = ' '.join(s.text.strip() for s in segs).strip()
    if len(text) >= 3:
        rows.append({'audio': w, 'text': text})
    if i % 50 == 0:
        print(i, '/', len(wavs))
print('kept', len(rows), 'labeled pairs')
json.dump(rows, open('/content/jahns_pairs.json','w'))

In [ ]:
# 5. Build a HF dataset (24 kHz audio + text). This is the object to reuse if you
#    fall back to Unsloth's official notebook.
from datasets import Dataset, Audio
ds = Dataset.from_list(rows).cast_column('audio', Audio(sampling_rate=24000))
ds.save_to_disk('/content/jahns_ds')
print(ds)
print(ds[0]['text'][:120])

In [ ]:
# 6. Load model + SNAC, define the Orpheus training-sequence format.
#    (Token scheme mirrors Unsloth's Orpheus recipe — cross-check if it drifts.)
from unsloth import FastLanguageModel
import torch, soundfile as sf, numpy as np
from snac import SNAC

model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/orpheus-3b-0.1-ft', max_seq_length=2048, dtype=None, load_in_4bit=False)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth')

snac = SNAC.from_pretrained('hubertsiuzdak/snac_24khz').eval().cuda()

# special tokens (Orpheus): human/ai/speech markers + SNAC code offset
TOK = 128256  # base offset for audio codes
SOH, EOH, SOA, SOS, EOS_SP, EOA = 128259, 128260, 128261, 128257, 128258, 128262

def encode_audio(wave):
    x = torch.tensor(wave, dtype=torch.float32).unsqueeze(0).unsqueeze(0).cuda()
    with torch.no_grad():
        codes = snac.encode(x)   # list of 3 codebooks
    # interleave 7 tokens/frame with per-layer offsets (Orpheus SNAC layout)
    ids = []
    for i in range(codes[0].shape[1]):
        ids += [
            TOK + codes[0][0][i] + 0*4096,
            TOK + codes[1][0][2*i] + 1*4096,
            TOK + codes[2][0][4*i] + 2*4096,
            TOK + codes[2][0][4*i+1] + 3*4096,
            TOK + codes[1][0][2*i+1] + 4*4096,
            TOK + codes[2][0][4*i+2] + 5*4096,
            TOK + codes[2][0][4*i+3] + 6*4096,
        ]
    return ids

def format_example(ex):
    wave = ex['audio']['array']
    audio_ids = encode_audio(np.asarray(wave))
    text_ids = tokenizer(ex['text'], add_special_tokens=False)['input_ids']
    input_ids = [SOH] + text_ids + [EOH] + [SOA, SOS] + audio_ids + [EOS_SP, EOA]
    return {'input_ids': input_ids, 'labels': input_ids, 'attention_mask': [1]*len(input_ids)}

proc = ds.map(format_example, remove_columns=ds.column_names)
print('formatted', len(proc), 'examples; sample len', len(proc[0]['input_ids']))

In [ ]:
# 7. Train (LoRA). ~1-2 h on a T4 for ~400 examples / 3 epochs.
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=proc,
    args=SFTConfig(
        per_device_train_batch_size=1, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        logging_steps=10, optim='adamw_8bit', lr_scheduler_type='linear',
        seed=42, output_dir='/content/orpheus-jahns', dataset_kwargs={'skip_prepare_dataset': True}),
)
trainer.train()

In [ ]:
# 8. Export a Q4 GGUF for your M1 (LM Studio, Metal). Also save LoRA + merged to Drive.
model.save_pretrained_merged('/content/orpheus-jahns-merged', tokenizer)
model.save_pretrained_gguf('/content/orpheus-jahns-gguf', tokenizer, quantization_method='q4_k_m')
!cp -r /content/orpheus-jahns-gguf /content/drive/MyDrive/
!ls -la /content/drive/MyDrive/orpheus-jahns-gguf/
print('Download the .gguf -> load in LM Studio on your M1 with the SNAC decoder.')

## 9. Play it on your M1

Orpheus emits SNAC audio tokens; decode them with `snac_24khz`. Use the official
[Orpheus inference](https://github.com/canopyai/Orpheus-TTS) glue (LM Studio serves the
GGUF over its OpenAI-compatible API; a small script pulls tokens → SNAC → 24 kHz WAV).

**Listen for:** (a) Jahns timbre, (b) whether his animated/sarcastic delivery carries or
flattens. That verdict → next step in `../report.md` §9 (sarcasm-curated subset or the
IndexTTS-2 Emo-Audio layer).